# Biotic Hardware Synthesis — Comparative Morphology Demo v1.4.0

Full pipeline: ten morphologies (fractal, botanical, random control, Fibonacci spiral, Voronoi control, hexagonal lattice, DLA, Gaussian clusters, concentric rings, reticulate vein growth), 3-metric statistics over 45 pairs, multi-seed distributions, and graph-topology analysis.

**Run from the repo root** (`biotic-hardware/`), not from inside `notebook/`. Generated artifacts are written to `outputs/`.

## 1 — Run the full pipeline

In [ ]:
import subprocess, sys

result = subprocess.run([sys.executable, 'run.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 2 — Morphological sensitivity curves (normalized)

Ten morphologies. Solid = Merit Scaled, dashed = Coherence Ratio.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

paths = {
    'Fractal':     'outputs/simulation_results_fractal.csv',
    'Botanical':   'outputs/simulation_results_botanical.csv',
    'Random':      'outputs/simulation_results_random.csv',
    'Fibonacci':   'outputs/simulation_results_fibonacci.csv',
    'Voronoi':     'outputs/simulation_results_voronoi.csv',
    'Hexagonal':   'outputs/simulation_results_hexagonal.csv',
    'DLA':         'outputs/simulation_results_dla.csv',
    'Clusters':    'outputs/simulation_results_clusters.csv',
    'Concentric':  'outputs/simulation_results_concentric.csv',
    'Reticulate':  'outputs/simulation_results_reticulate.csv',
}
colors = {
    'Fractal':    '#2196F3',
    'Botanical':  '#4CAF50',
    'Random':     '#FF5722',
    'Fibonacci':  '#9C27B0',
    'Voronoi':    '#FF9800',
    'Hexagonal':  '#00BCD4',
    'DLA':        '#795548',
    'Clusters':   '#E91E63',
    'Concentric': '#607D8B',
    'Reticulate': '#8BC34A',
}

def norm(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-12)

fig, ax = plt.subplots(figsize=(15, 7))
for name, path in paths.items():
    df = pd.read_csv(path)
    ax.plot(df['Distance'], norm(df['Merit_Scaled']), label=f'{name} - Merit Scaled',
            linewidth=2, color=colors[name])
    ax.plot(df['Distance'], norm(df['Coherence_Ratio']), label=f'{name} - Coherence',
            linewidth=1.1, color=colors[name], linestyle='--', alpha=0.55)

ax.set_xlabel('Distance (m)')
ax.set_ylabel('Normalized value')
ax.set_title('Morphological Sensitivity Benchmark v1.4.0 - Ten Morphologies (seed 42)')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 3 — Statistical separation: 3 metrics x 45 pairs

Welch t-test + Cohen's d read from `outputs/curve_separation_summary.csv` (generated by the pipeline).  
**p-value**: green = significant. **|Cohen's d|**: blue = large effect.

In [ ]:
df_stat = pd.read_csv('outputs/curve_separation_summary.csv')

metrics = ['Merit_Scaled', 'Coherence_Ratio', 'Peak_AF']
pairs = sorted(df_stat['Pair'].unique())

p_matrix = np.full((len(pairs), len(metrics)), np.nan)
d_matrix = np.full((len(pairs), len(metrics)), np.nan)

for _, row in df_stat.iterrows():
    if row['Metric'] not in metrics or row['Pair'] not in pairs:
        continue
    i = pairs.index(row['Pair'])
    j = metrics.index(row['Metric'])
    p_matrix[i, j] = row['p_value']
    d_matrix[i, j] = abs(row['Cohens_d'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 15))

im1 = ax1.imshow(p_matrix, cmap='RdYlGn_r', vmin=0, vmax=0.1, aspect='auto')
ax1.set_xticks(range(len(metrics)))
ax1.set_xticklabels(metrics, rotation=20, ha='right', fontsize=8)
ax1.set_yticks(range(len(pairs)))
ax1.set_yticklabels(pairs, fontsize=6)
ax1.set_title('p-value (green = significant, threshold 0.05)')
for i in range(len(pairs)):
    for j in range(len(metrics)):
        v = p_matrix[i, j]
        if np.isnan(v):
            continue
        ax1.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=5,
                 color='white' if v < 0.01 else 'black')
plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

d_max = max(np.nanmax(d_matrix), 1.0)
im2 = ax2.imshow(d_matrix, cmap='Blues', vmin=0, vmax=d_max, aspect='auto')
ax2.set_xticks(range(len(metrics)))
ax2.set_xticklabels(metrics, rotation=20, ha='right', fontsize=8)
ax2.set_yticks(range(len(pairs)))
ax2.set_yticklabels(pairs, fontsize=6)
ax2.set_title("|Cohen's d| (dark blue = large effect >0.8)")
for i in range(len(pairs)):
    for j in range(len(metrics)):
        v = d_matrix[i, j]
        if np.isnan(v):
            continue
        ax2.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=5,
                 color='white' if v > d_max * 0.6 else 'black')
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

fig.suptitle('Statistical Separation - Welch t-test + Cohen d (v1.4.0) - 3 metrics x 45 pairs',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

df_stat[['Metric', 'Pair', 'p_value', 'Significant_p05', 'Cohens_d', 'Effect_size']]

## 4 — Multi-seed analysis: distributions by morphology (seeds 42–71)

Each bar is the mean over 30 seeds. Error bars = ±1 std.  
Under the v1.4.0 sector phase rule the positional noise shifts nodes across sector boundaries, so all ten morphologies — including the geometrically deterministic fractal, Fibonacci and hexagonal — carry genuine per-seed variance, and no pair is dropped as degenerate.

In [ ]:
df_multi = pd.read_csv('outputs/multi_seed_summary.csv')

morphologies = df_multi['Morphology'].tolist()
colors = {
    'fractal':    '#2196F3',
    'botanical':  '#4CAF50',
    'random':     '#FF5722',
    'fibonacci':  '#9C27B0',
    'voronoi':    '#FF9800',
    'hexagonal':  '#00BCD4',
    'dla':        '#795548',
    'clusters':   '#E91E63',
    'concentric': '#607D8B',
    'reticulate': '#8BC34A',
}

metrics_multi = [
    ('Merit_Mean',     'Merit_Std',     'Merit Scaled'),
    ('Coherence_Mean', 'Coherence_Std', 'Coherence Ratio'),
    ('PeakAF_Mean',    'PeakAF_Std',    'Peak AF'),
]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, (mean_col, std_col, label) in zip(axes, metrics_multi):
    means = df_multi[mean_col].values
    stds  = df_multi[std_col].values
    bar_colors = [colors.get(m, '#888888') for m in morphologies]
    bars = ax.bar(morphologies, means, yerr=stds, color=bar_colors,
                  capsize=5, alpha=0.85, width=0.7)
    for bar, mean, std in zip(bars, means, stds):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            mean + std + max(means) * 0.03,
            f'{mean:.3f}',
            ha='center', va='bottom', fontsize=6
        )
    ax.set_title(label)
    ax.set_ylabel('Value')
    ax.tick_params(axis='x', rotation=40)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Multi-seed Sensitivity Analysis - seeds [42...71] (v1.4.0) - Ten Morphologies',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5 — Graph topology

k-nearest-neighbour graphs per morphology; spectral descriptors averaged over 30 seeds at the primary `k=6`. The topology-vs-merit correlations (H1 eigenratio R, H2 algebraic connectivity lambda_2, H3 clustering) are **exploratory** — none reach the pre-specified criterion (|r| >= 0.632 and p < 0.05) at N=10.

In [ ]:
df_topo = pd.read_csv('outputs/graph_topology_summary.csv')
df_topo_k6 = df_topo[df_topo['k'] == 6].sort_values('Merit_Mean').reset_index(drop=True)

topo_colors = {
    'fractal':    '#2196F3',
    'botanical':  '#4CAF50',
    'random':     '#FF5722',
    'fibonacci':  '#9C27B0',
    'voronoi':    '#FF9800',
    'hexagonal':  '#00BCD4',
    'dla':        '#795548',
    'clusters':   '#E91E63',
    'concentric': '#607D8B',
    'reticulate': '#8BC34A',
}

descriptors = [
    ('lambda_2_mean',   'lambda_2_std',   'Algebraic connectivity (lambda_2)'),
    ('eigenratio_mean', 'eigenratio_std', 'Eigenratio R'),
    ('clustering_mean', 'clustering_std', 'Clustering coefficient'),
]

order = df_topo_k6['Morphology'].tolist()
bar_colors = [topo_colors.get(m, '#888888') for m in order]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (mean_col, std_col, label) in zip(axes, descriptors):
    ax.bar(order, df_topo_k6[mean_col].values, yerr=df_topo_k6[std_col].values,
           color=bar_colors, capsize=5, alpha=0.85, width=0.7)
    ax.set_title(label)
    ax.tick_params(axis='x', rotation=40)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Graph topology at k=6 - spectral descriptors over 30 seeds (v1.4.0)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

df_corr = pd.read_csv('outputs/topology_correlation.csv')
df_corr[df_corr['k'] == 6][['Hypothesis', 'Metric', 'Target', 'r', 'p', 'threshold_abs_r', 'passes_threshold', 'loocv_sign_stable']]

## 6 — exploration_summary.json

Machine-readable output: experimental configuration + multi-seed results in a single structured entry point.

In [ ]:
with open('outputs/exploration_summary.json') as f:
    summary = json.load(f)

print(json.dumps(summary, indent=2))